In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split

from _load_dataset import load_dataset_sparse_labels

In [ ]:
# inclusive range of seeds
SEED_RANGE = (0, 1000)

# choose to minimize or maximize MSE
DIRECTION = "maximize"  # "minimize" or "maximize"

In [ ]:
# load data
_, s008_lidar, _, _, s009_lidar, _ = load_dataset_sparse_labels()

# prepare output folder
out_dir = "runs/pca_seed"
os.makedirs(out_dir, exist_ok=True)

In [ ]:
s008_flat = s008_lidar.reshape(s008_lidar.shape[0], -1)
s009_flat = s009_lidar.reshape(s009_lidar.shape[0], -1)

In [ ]:
results = []
for seed in range(SEED_RANGE[0], SEED_RANGE[1] + 1):
    # split data
    data008, _ = train_test_split(s008_flat, train_size=0.8, random_state=seed)
    _, data009 = train_test_split(s009_flat, test_size=0.2, random_state=seed)

    # fit PCA
    pca8 = PCA(n_components=3, random_state=seed)
    pca9 = PCA(n_components=3, random_state=seed)
    pca8.fit(data008)
    pca9.fit(data009)

    # compute explained variance difference
    ev8 = pca8.explained_variance_ratio_
    ev9 = pca9.explained_variance_ratio_
    diff = ev8 - ev9

    # compute metrics
    l1 = np.sum(np.abs(diff))
    l2 = np.sqrt(np.sum(diff**2))
    linf = np.max(np.abs(diff))
    mse = np.mean(diff**2)
    results.append((seed, l1, l2, linf, mse))

    # project data
    proj008 = pca8.transform(data008)
    proj009 = pca9.transform(data009)

    # plot 3D scatter
    fig = plt.figure()
    ax = fig.add_subplot(111, projection="3d")
    ax.scatter(proj008[:, 0], proj008[:, 1], proj008[:, 2], label="s008", alpha=0.5)
    ax.scatter(proj009[:, 0], proj009[:, 1], proj009[:, 2], label="s009", alpha=0.5)
    ax.set_title(f"Seed {seed}  " f"L1={l1:.3f}  L2={l2:.3f}  Linf={linf:.3f}  MSE={mse:.6f}")
    ax.set_xlabel("PC1")
    ax.set_ylabel("PC2")
    ax.set_zlabel("PC3")
    ax.legend()

    # save plot
    fname = f"seed_{seed}_" f"L1_{l1:.3f}_" f"L2_{l2:.3f}_" f"Linf_{linf:.3f}_" f"MSE_{mse:.6f}.png"
    fig.savefig(os.path.join(out_dir, fname))
    plt.close(fig)
    
    print (f"Processed seed {seed}: L1={l1:.3f}, L2={l2:.3f}, Linf={linf:.3f}, MSE={mse:.6f}")

In [ ]:
# print metrics table
print("L1 -> L1 norm, this sums the absolute per-component differences.")
print("L2 -> L2 norm, this is the Euclidean distance between the two vectors.")
print("Linf -> Linf norm, this is the maximum absolute per-component difference.")
print("MSE -> Mean Squared Error, this is the average of the squared differences.\n")

print("seed   L1       L2       Linf     MSE")
for seed, l1, l2, linf, mse in results:
    print(f"{seed:>4} {l1:>8.3f} {l2:>8.3f} {linf:>8.3f} {mse:>10.6f}")

# select best seeds for each metric based on DIRECTION
if DIRECTION == "minimize":
    best_l1 = min(results, key=lambda x: x[1])
    best_l2 = min(results, key=lambda x: x[2])
    best_linf = min(results, key=lambda x: x[3])
    best_mse = min(results, key=lambda x: x[4])
else:
    best_l1 = max(results, key=lambda x: x[1])
    best_l2 = max(results, key=lambda x: x[2])
    best_linf = max(results, key=lambda x: x[3])
    best_mse = max(results, key=lambda x: x[4])

# print summary
print(f"Best seed for {DIRECTION} L1: {best_l1[0]} (L1 = {best_l1[1]:.3f})")
print(f"Best seed for {DIRECTION} L2: {best_l2[0]} (L2 = {best_l2[2]:.3f})")
print(f"Best seed for {DIRECTION} Linf: {best_linf[0]} (Linf = {best_linf[3]:.3f})")
print(f"Best seed for {DIRECTION} MSE: {best_mse[0]} (MSE = {best_mse[4]:.6f})")